In [ ]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [ ]:
df = pd.read_csv('./Churn_Modelling.csv')
df.head()

In [ ]:
df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
label_encoder_gender = LabelEncoder()
df['Gender'] = label_encoder_gender.fit_transform(df['Gender'])
df

In [ ]:
#one-hot encoding for geography
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder = OneHotEncoder()
geography_encoded = one_hot_encoder.fit_transform(df[['Geography']])

In [ ]:
geography_encoded

In [ ]:
geography_encoded_df = pd.DataFrame(geography_encoded.toarray(), columns=one_hot_encoder.get_feature_names_out(['Geography']))

In [ ]:
geography_encoded_df

In [ ]:
## combine one hot encoded column w original dataset
df = pd.concat([df.drop('Geography', axis=1), geography_encoded_df], axis=1)
df.head()

In [ ]:
## Save the encoders for future use

with open('label_encoder_gender.pkl', 'wb') as f:
    pickle.dump(label_encoder_gender, f)
with open('one_hot_encoder_geography.pkl', 'wb') as f:
    pickle.dump(one_hot_encoder, f)


In [ ]:
## Divide the dataset in independent and dependent features
X = df.drop('Exited', axis=1)
y = df['Exited']

## Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train.shape

In [ ]:
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

ANN IMPLEMENTATION

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [ ]:
## Builing ANN model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [ ]:
model.summary()

In [ ]:
opt = tf.keras.optimizers.legacy.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy()

In [ ]:
#Compile the model
model.compile(optimizer= opt, loss=loss, metrics=['accuracy'])

In [ ]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [ ]:
## Set up early stopping
early_stopping_callback = EarlyStopping(monitor = 'val_loss', patience=5, restore_best_weights=True)

In [ ]:
history = model.fit(
    X_train, y_train, validation_data=(X_test, y_test), epochs=100,
    callbacks=[tensorflow_callback, early_stopping_callback]
)

In [ ]:
model.save('model.h5')

In [ ]:
#Load TensorBoard extension
%load_ext tensorboard

In [ ]:
|## Load the pickle file


